::: {.sb-downloads}
[Download this notebook](../downloads/solution-walmart-sales.ipynb){download="solution-walmart-sales.ipynb"}  
Data: [walmart_store_sales.csv](../downloads/data/walmart_store_sales.csv){download="walmart_store_sales.csv"}

Put the data files in a folder named `data` next to the notebook.
:::

::: {.sb-lesson-links}
[← Back to the project](../lessons/walmart-sales.ipynb)
:::

::: {.callout-tip}
Work on the project on your own first: you learn much more by searching than by reading a solution.
:::

## Part 1 - EDA and preprocessing

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score

# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

In [ ]:
dataset = pd.read_csv("data/walmart_store_sales.csv")
print("Number of rows:", dataset.shape[0])
dataset.head()

Number of rows: 150


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092


In [ ]:
print("Percentage of missing values:")
(100 * dataset.isnull().sum() / dataset.shape[0]).round(1)

Percentage of missing values:


Store            0.0
Date            12.0
Weekly_Sales     9.3
Holiday_Flag     8.0
Temperature     12.0
Fuel_Price       9.3
CPI              8.0
Unemployment    10.0
dtype: float64

In [ ]:
dataset.describe(include="all")

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,132,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,07-01-2011,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,NaN,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000


**A small dataset (150 rows) with missing values in every column except *Store*. 20 different stores appear in the data.**

In [ ]:
# Distribution of each numeric variable
num_features = ["Weekly_Sales", "Temperature", "Fuel_Price", "CPI", "Unemployment"]
fig = make_subplots(rows=len(num_features), cols=1, subplot_titles=num_features)
for i, f in enumerate(num_features):
    fig.add_trace(go.Histogram(x=dataset[f], nbinsx=50), row=i + 1, col=1)
fig.update_layout(title="Distributions of the numeric variables", showlegend=False, height=1200)
fig.show()

**The distributions are far from normal: *CPI* has two groups of values (the stores are in regions with different price levels), and *Weekly_Sales* is spread over a wide range. With so few rows, we keep the variables as they are.**

In [ ]:
# Bivariate analysis: target vs. each numeric feature
for f in num_features[1:]:
    fig = px.scatter(dataset, x=f, y="Weekly_Sales", title=f"Weekly sales vs. {f}")
    fig.show()

In [ ]:
# Weekly sales of each store
fig = px.box(dataset, x="Store", y="Weekly_Sales", title="Weekly sales by store")
fig.show()

**No numeric feature shows a strong link with the sales on its own. The *Store*, on the other hand, changes everything: the typical weekly sales of the biggest store are about seven times those of the smallest one (bigger stores, different areas). The model will need this information.**

### Dates
⚠️ The dates are written day first (`18-02-2011`). Without the `format` argument, pandas could read some of them month first (`05-03-2011` would become 3 May instead of 5 March), and silently produce wrong features.

In [ ]:
dataset["Date"] = pd.to_datetime(dataset["Date"], format="%d-%m-%Y")
print("From", dataset["Date"].min().date(), "to", dataset["Date"].max().date())
print("Days of the week:", dataset["Date"].dt.day_name().value_counts().to_dict())

fig = px.scatter(dataset.sort_values("Date"), x="Date", y="Weekly_Sales", color=dataset.sort_values("Date")["Store"].astype(str),
                 title="Weekly sales over time, by store")
fig.show()

From 2010-02-05 to 2012-10-19
Days of the week: {'Friday': 132}


**All the dates are Fridays (the end of each sales week), so the day of the week is useless. We'll extract the year, the month and the day.**

### Preprocessing with pandas

* drop the rows where the target is missing (we never impute the target: it would bias the model);
* create features from the *Date* column, then drop it;
* drop the outliers: the values outside $[\bar{X} - 3\sigma, \bar{X} + 3\sigma]$ for *Temperature*, *Fuel_Price*, *CPI* and *Unemployment*.

In [ ]:
dataset = dataset.loc[dataset["Weekly_Sales"].notnull(), :]

dataset["Year"] = dataset["Date"].dt.year
dataset["Month"] = dataset["Date"].dt.month
dataset["Day"] = dataset["Date"].dt.day
dataset = dataset.drop("Date", axis=1)


def drop_outliers(df, col_name):
    """Keep the rows where the value is missing or within 3 standard deviations of the mean."""
    mean, std = df[col_name].mean(), df[col_name].std()
    to_keep = df[col_name].isnull() | ((df[col_name] > mean - 3 * std) & (df[col_name] < mean + 3 * std))
    return df.loc[to_keep, :]


for col in ["Temperature", "Fuel_Price", "CPI", "Unemployment"]:
    dataset = drop_outliers(dataset, col)
print("Rows remaining:", dataset.shape[0])
dataset.head()

Rows remaining: 131


,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Month,Day
0,6.0,1572117.54,NaN,59.61,3.045,214.777523,6.858,2011.0,2.0,18.0
1,13.0,1807545.43,0.0,42.38,3.435,128.616064,7.470,2011.0,3.0,25.0
3,11.0,1244390.03,0.0,84.57,NaN,214.556497,7.346,NaN,NaN,NaN
4,6.0,1644470.66,0.0,78.89,2.759,212.412888,7.092,2010.0,5.0,28.0
5,4.0,1857533.70,0.0,NaN,2.756,126.160226,7.896,2010.0,5.0,28.0


### Preprocessing with scikit-learn

* categorical features: *Store* and *Holiday_Flag* → impute the most frequent value, then one-hot encode;
* numeric features → impute the mean, then standardize.

In [ ]:
target_name = "Weekly_Sales"
X = dataset.drop(target_name, axis=1)
Y = dataset[target_name]

categorical_features = ["Store", "Holiday_Flag"]
numeric_features = ["Temperature", "Fuel_Price", "CPI", "Unemployment", "Year", "Month", "Day"]

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("Train:", X_train.shape, " Test:", X_test.shape)

Train: (104, 9)  Test: (27, 9)


In [ ]:
preprocessor = ColumnTransformer(transformers=[
    ("num", Pipeline([("imputer", SimpleImputer(strategy="mean")), ("scaler", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("encoder", OneHotEncoder(drop="first", handle_unknown="ignore"))]), categorical_features),
])

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)  # don't fit again!
feature_names = preprocessor.get_feature_names_out()
print(len(feature_names), "features:", list(feature_names))

26 features: ['num__Temperature', 'num__Fuel_Price', 'num__CPI', 'num__Unemployment', 'num__Year', 'num__Month', 'num__Day', 'cat__Store_2.0', 'cat__Store_3.0', 'cat__Store_4.0', 'cat__Store_5.0', 'cat__Store_6.0', 'cat__Store_7.0', 'cat__Store_8.0', 'cat__Store_9.0', 'cat__Store_10.0', 'cat__Store_11.0', 'cat__Store_13.0', 'cat__Store_14.0', 'cat__Store_15.0', 'cat__Store_16.0', 'cat__Store_17.0', 'cat__Store_18.0', 'cat__Store_19.0', 'cat__Store_20.0', 'cat__Holiday_Flag_1.0']


## Part 2 - Baseline model (linear regression)

In [ ]:
model = LinearRegression()
model.fit(X_train, Y_train)

print("R2 score on training set:", round(r2_score(Y_train, model.predict(X_train)), 3))
print("R2 score on test set:    ", round(r2_score(Y_test, model.predict(X_test)), 3))

R2 score on training set: 0.973
R2 score on test set:     0.934


**The baseline already explains most of the variance of the sales ($R^2 \approx 0.93$ on the test set). The score is a bit lower on the test set than on the training set: the model overfits a little, which is not surprising with 26 features for about 100 training rows.**

### Importance of the features
The features are standardized (or 0/1 for the one-hot columns), so we can compare the absolute values of the coefficients.

In [ ]:
model_coefficients = pd.DataFrame(index=feature_names, data={"baseline": model.coef_})
model_coefficients.sort_values(by="baseline", key=np.abs, ascending=False).round(0)

,baseline
cat__Store_5.0,-1402927.0
cat__Store_9.0,-1268176.0
cat__Store_3.0,-1251809.0
cat__Store_16.0,-1107620.0
cat__Store_7.0,-889546.0
cat__Store_8.0,-754920.0
cat__Store_14.0,722199.0
cat__Store_10.0,677716.0
cat__Store_15.0,-672368.0
cat__Store_17.0,-667730.0


In [ ]:
numeric = model_coefficients.loc[[f"num__{f}" for f in numeric_features], "baseline"].sort_values()
fig = px.bar(x=numeric.values, y=numeric.index, orientation="h", title="Coefficients of the numeric features",
             labels={"x": "effect on the weekly sales ($) for +1 standard deviation", "y": ""})
fig.show()

**The most important information is the *Store*: the one-hot columns of the stores have by far the largest coefficients (up to more than 1 million $ per week compared to the reference store). This makes sense, as the size and the location of a store largely determine its sales.**

**Among the other features, *Unemployment* has the largest coefficient, and it is negative: the higher the unemployment rate, the lower the sales. *CPI* and *Month* come next (sales grow at the end of the year, with the holiday season). *Temperature*, *Fuel_Price* and *Day* matter much less.**

## Part 3 - Fight overfitting
### Ridge

In [ ]:
ridge = Ridge()  # default regularization strength: alpha = 1
ridge.fit(X_train, Y_train)
print("R2 score on training set:", round(ridge.score(X_train, Y_train), 3))
print("R2 score on test set:    ", round(ridge.score(X_test, Y_test), 3))

R2 score on training set: 0.933
R2 score on test set:     0.925


The default value of $\alpha$ is not necessarily a good one. Let's tune it with a grid search. Regularization strengths are usually tried on a **logarithmic scale** (0.001, 0.01, 0.1, 1, 10...): `np.logspace` builds such a grid.

In [ ]:
ridge_cv = GridSearchCV(Ridge(), {"alpha": np.logspace(-3, 3, 25)}, cv=5)
ridge_cv.fit(X_train, Y_train)
print("Best alpha:", ridge_cv.best_params_["alpha"])
print("R2 score on training set:", round(ridge_cv.score(X_train, Y_train), 3))
print("R2 score on test set:    ", round(ridge_cv.score(X_test, Y_test), 3))
model_coefficients["Ridge"] = ridge_cv.best_estimator_.coef_

Best alpha: 0.001
R2 score on training set: 0.973
R2 score on test set:     0.935


**The grid search chooses the smallest $\alpha$ of the grid: Ridge regularization does not help here, and the tuned model is almost the same as the linear regression.**

### Lasso

In [ ]:
lasso_cv = GridSearchCV(Lasso(max_iter=50000), {"alpha": np.logspace(0, 5, 26)}, cv=5)
lasso_cv.fit(X_train, Y_train)
print("Best alpha:", round(lasso_cv.best_params_["alpha"], 1))
print("R2 score on training set:", round(lasso_cv.score(X_train, Y_train), 3))
print("R2 score on test set:    ", round(lasso_cv.score(X_test, Y_test), 3))
model_coefficients["Lasso"] = lasso_cv.best_estimator_.coef_
print("Coefficients equal to zero:", (lasso_cv.best_estimator_.coef_ == 0).sum())

Best alpha: 158.5
R2 score on training set: 0.973
R2 score on test set:     0.938
Coefficients equal to zero: 0


In [ ]:
model_coefficients.sort_values(by="baseline", key=np.abs, ascending=False).round(0)

,baseline,Ridge,Lasso
cat__Store_5.0,-1402927.0,-1401472.0,-1385429.0
cat__Store_9.0,-1268176.0,-1266384.0,-1244981.0
cat__Store_3.0,-1251809.0,-1250708.0,-1240393.0
cat__Store_16.0,-1107620.0,-1106304.0,-1096213.0
cat__Store_7.0,-889546.0,-889248.0,-896084.0
cat__Store_8.0,-754920.0,-753627.0,-738974.0
cat__Store_14.0,722199.0,722164.0,707298.0
cat__Store_10.0,677716.0,677317.0,645041.0
cat__Store_15.0,-672368.0,-671942.0,-686519.0
cat__Store_17.0,-667730.0,-666724.0,-670914.0


## Conclusion

* A linear model predicts the weekly sales well ($R^2 \approx 0.93$ on the test set). The *Store* is by far the most important feature, then the economic indicators (*Unemployment*, negatively, and *CPI*) and the time of year.
* Regularization brings little here: the tuned Lasso improves the test score only slightly, and Ridge not at all. The baseline was only mildly overfitting.
* Be careful with conclusions drawn from such a small dataset: the test set has only 27 rows, so a single train/test split gives a noisy score. Cross-validated scores (`best_score_` of the grid search) are more reliable to compare models.

To go further, a model could use more history for each store (past sales are often the best predictors of future sales): that's where time series models come in.